# Master Run Notebook

Executes every experiment notebook in the correct order and reports pass/fail for each.

---

## Experiment Flow

```
RAW DATA (EOS-04 + Sentinel-1 CSVs)
        │
        ▼
┌─────────────────────────────────────────────────────┐
│  STAGE 1 — Data Exploration                         │
│  Visualise distributions, correlations, censoring   │
└─────────────────────┬───────────────────────────────┘
                      │
        ┌─────────────┴─────────────┐
        │                           │
        ▼                           ▼
   CENSORED data              UNCENSORED data
  (keeps SM=50 rows)         (drops SM=50 rows)
        │                           │
        ▼                           ▼
┌───────────────────────────────────────────────────────────┐
│  STAGE 2 — Classification                                 │
│  Discretise SM into classes → RF / XGB / AdaBoost / SVC  │
└───────────────────────────────────────────────────────────┘
        │
        ▼
┌───────────────────────────────────────────────────────────┐
│  STAGE 3 — Classical ML Regression  (baseline)           │
│  RF / XGB / AdaBoost / SVR  →  MAE, RMSE, R²             │
└───────────────────────────────────────────────────────────┘
        │
        ▼
┌───────────────────────────────────────────────────────────┐
│  STAGE 4 — ANN Regression  (point predictions)           │
│  Dense networks with MSE loss  →  MAE, R²                │
└───────────────────────────────────────────────────────────┘
        │
        ▼
┌───────────────────────────────────────────────────────────┐
│  STAGE 5 — Prediction Intervals: Quantile Regression     │
│  Two ANNs (τ=0.025 / τ=0.975)  →  PICP, MPIW            │
└───────────────────────────────────────────────────────────┘
        │
        ▼
┌───────────────────────────────────────────────────────────┐
│  STAGE 6 — Conformal Prediction                          │
│  MAPIE CQR on QuantileReg / GBR / HistGBR               │
└───────────────────────────────────────────────────────────┘
        │
        ▼
┌───────────────────────────────────────────────────────────┐
│  STAGE 7 — CQR + Tau Tuning  (uncensored only)          │
│  ANN CQR + SVM conformal + sweep τ pairs                 │
└───────────────────────────────────────────────────────────┘
        │
        ▼
┌───────────────────────────────────────────────────────────┐
│  STAGE 8 — SVR Hyperparameter Tuning  (uncensored only) │
│  Quantile SVR grid search over C and gamma               │
└───────────────────────────────────────────────────────────┘
```

> **Both EOS-04 and Sentinel-1 satellites are run inside every notebook.**  
> Exploration notebooks require the raw `.xlsx` files — they are skipped if the files are not present.


In [ ]:
import subprocess
import sys
import time
from pathlib import Path
from IPython.display import display, HTML

CODE_DIR = Path.cwd()
results = []

def run_notebook(name: str, skip_if_missing: list[str] = None) -> dict:
    """
    Execute a notebook in-place via nbconvert.
    Returns a result dict with name, status, elapsed time.
    """
    nb_path = CODE_DIR / f"{name}.ipynb"

    # Optional: skip if prerequisite files are absent
    if skip_if_missing:
        for f in skip_if_missing:
            if not Path(f).exists():
                print(f"⏭  SKIPPED  {name}  (missing: {f})")
                return {"name": name, "status": "SKIPPED", "elapsed": 0}

    print(f"▶  Running  {name} ...", end="", flush=True)
    start = time.time()

    proc = subprocess.run(
        [
            sys.executable, "-m", "nbconvert",
            "--to", "notebook",
            "--execute",
            "--inplace",
            "--ExecutePreprocessor.timeout=7200",
            "--ExecutePreprocessor.kernel_name=python3",
            str(nb_path),
        ],
        capture_output=True,
        text=True,
        cwd=str(CODE_DIR),
    )

    elapsed = time.time() - start
    ok = proc.returncode == 0
    status = "PASSED" if ok else "FAILED"
    print(f"  {'✓' if ok else '✗'}  {status}  ({elapsed/60:.1f} min)")

    if not ok:
        # Print last 600 chars of stderr for quick diagnosis
        tail = (proc.stderr or proc.stdout or "")[-600:]
        print(f"  └─ {tail}")

    entry = {"name": name, "status": status, "elapsed_min": round(elapsed / 60, 1)}
    results.append(entry)
    return entry

print("Setup complete. CODE_DIR:", CODE_DIR)

---
## Stage 1 — Data Exploration

Visualises the raw satellite data: distributions of backscatter values, soil moisture histograms, correlation heatmaps, and the extent of censoring (SM = 50 rows).  
**Requires the raw `.xlsx` files** — skipped automatically if they are not present in `data/`.

| Notebook | Satellite | Purpose |
|---|---|---|
| `exploration_eos.ipynb` | EOS-04 | Distribution plots, HH/HV correlations |
| `exploration_sentinel.ipynb` | Sentinel-1 | Distribution plots, VH/VV correlations |

In [ ]:
DATA_DIR = CODE_DIR.parent / "data"

run_notebook("exploration_eos",      skip_if_missing=[str(DATA_DIR / "EOS-04_datasheet.xlsx")])
run_notebook("exploration_sentinel", skip_if_missing=[str(DATA_DIR / "sentinel-1.xlsx")])

---
## Stage 2 — Classification

Converts soil moisture values into discrete moisture classes and runs four classifiers (Random Forest, XGBoost, AdaBoost, SVC).  
Produces a JSON classification report (accuracy + F1 per class) for each satellite.

**Censored** = includes rows where SM = 50 (sensor detection limit hit).  
**Uncensored** = drops those rows before training.

| Notebook | Data | Output folder |
|---|---|---|
| `classification_censored.ipynb` | All rows | `output/classification_censored/` |
| `classification_uncensored.ipynb` | SM ≠ 50 only | `output/classification_uncensored/` |

In [ ]:
run_notebook("classification_censored")
run_notebook("classification_uncensored")

---
## Stage 3 — Classical ML Regression  *(baseline)*

Trains four regression models (RF, XGBoost, AdaBoost, SVR) with 3-fold grid search on each.  
Establishes a performance baseline for comparing neural and PI methods.

**Metrics saved**: MAE, RMSE, R², MAPE  

| Notebook | Data | Output folder |
|---|---|---|
| `classical_ml_censored.ipynb` | All rows | `output/ml_experiment_censored/` |
| `classical_ml_uncensored.ipynb` | SM ≠ 50 only | `output/ml_experiment_uncensored/` |

In [ ]:
run_notebook("classical_ml_censored")
run_notebook("classical_ml_uncensored")

---
## Stage 4 — ANN Regression  *(point predictions)*

Trains dense neural networks with MSE loss. Multiple architectures are tested (varying depth, dropout).  
Produces a prediction-error scatter plot (actual vs predicted) for each model × satellite combination.  
Best architectures from this stage inform the PI estimation stages.

**Metrics saved**: MAE, MSE, R²  

| Notebook | Data | Output folder |
|---|---|---|
| `ann_censored.ipynb` | All rows | `output/ann_experiments_censored/` |
| `ann_uncensored.ipynb` | SM ≠ 50 only | `output/ann_experiments_uncensored/` |

In [ ]:
run_notebook("ann_censored")
run_notebook("ann_uncensored")

---
## Stage 5 — Prediction Intervals: Quantile Regression

Trains **two separate ANNs** per experiment — one for the lower bound (τ = 0.025) and one for the upper bound (τ = 0.975) — using pinball loss.  
Together they produce a 95% prediction interval `[lower, upper]` around the soil moisture estimate.

**Metrics saved**: PICP (coverage %), MPIW (average interval width)  
Target: PICP ≥ 95%, MPIW as small as possible.

| Notebook | Data | Output folder |
|---|---|---|
| `pi_estimation_censored.ipynb` | All rows | `output/pi_estimation_censored/` |
| `pi_estimation_uncensored.ipynb` | SM ≠ 50 only | `output/pi_estimation_uncensored/` |

In [ ]:
run_notebook("pi_estimation_censored")
run_notebook("pi_estimation_uncensored")

---
## Stage 6 — Conformal Prediction

Uses MAPIE's `ConformalizedQuantileRegressor` — a distribution-free method with **mathematically guaranteed** 95% coverage.  
Splits data into train / calibration / test. Three underlying models are compared: QuantileRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor.

**Metrics saved**: PICP, MPIW (per model, per satellite)  

| Notebook | Data | Output folder |
|---|---|---|
| `conformal_regression_censored.ipynb` | All rows | `output/conformal_regression_censored/` |
| `conformal_regression_uncensored.ipynb` | SM ≠ 50 only | `output/conformal_regression_uncensored/` |

In [ ]:
run_notebook("conformal_regression_censored")
run_notebook("conformal_regression_uncensored")

---
## Stage 7 — Conformalized Quantile Regression + Tau Tuning  *(uncensored only)*

Two advanced notebooks that build on the quantile ANN from Stage 5:

**`conformalized_quantile_regression_uncensored`** — Trains ANN quantile models, then applies CQR calibration (adjusts interval width using validation non-conformity scores), and also runs an SVM split-conformal baseline. Produces calibrated vs uncalibrated PICP/MPIW comparison.

**`quantile_regression_tau_tuning_uncensored`** — Sweeps multiple (τ_lower, τ_upper) pairs (all with 0.95 gap) to find the optimal quantile targets. Runs both raw QR and CQR-calibrated for each pair.

| Notebook | Output folder |
|---|---|
| `conformalized_quantile_regression_uncensored.ipynb` | `output/conformal_results_uncensored/` |
| `quantile_regression_tau_tuning_uncensored.ipynb` | `output/conformal_results_uncensored/` |

In [ ]:
run_notebook("conformalized_quantile_regression_uncensored")
run_notebook("quantile_regression_tau_tuning_uncensored")

---
## Stage 8 — SVR Hyperparameter Tuning  *(uncensored only)*

Runs a grid search over the Quantile SVR's `C` and `gamma` parameters using a custom inline `QuantileSVRExperiment` class (not from `model_experiments.py`).  
Finds the best kernel parameters for both lower and upper quantile SVR models.

| Notebook | Output folder |
|---|---|
| `quantile_svr_HP_tuning.ipynb` | `output/quantile_svr/` |

In [ ]:
run_notebook("quantile_svr_HP_tuning")

---
## Summary

In [ ]:
if not results:
    print("No notebooks have been run yet.")
else:
    col_w = max(len(r["name"]) for r in results) + 2
    header = f"{'Notebook':<{col_w}}  {'Status':<8}  {'Time (min)':>10}"
    sep    = "-" * len(header)
    print(sep)
    print(header)
    print(sep)
    for r in results:
        icon = "✓" if r["status"] == "PASSED" else ("⏭" if r["status"] == "SKIPPED" else "✗")
        print(f"{r['name']:<{col_w}}  {icon} {r['status']:<7}  {r['elapsed_min']:>10.1f}")
    print(sep)

    passed  = sum(1 for r in results if r["status"] == "PASSED")
    failed  = sum(1 for r in results if r["status"] == "FAILED")
    skipped = sum(1 for r in results if r["status"] == "SKIPPED")
    total_min = sum(r["elapsed_min"] for r in results)

    print(f"\nTotal: {passed} passed  |  {failed} failed  |  {skipped} skipped  |  {total_min:.1f} min")